In [3]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("rawData.csv", encoding="latin1")

# Limpiar nombres de columnas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
    .str.strip("_")
)

df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")
df["world_cup_year"] = df["tournament_id"].str.extract(r"(\d{4})").astype(int)

text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

df["goal_difference"] = df["home_team_score"] - df["away_team_score"]
df["total_goals"] = df["home_team_score"] + df["away_team_score"]

# 2 = gana local, 1 = empate, 0 = gana visitante
df["result_label"] = np.select(
    [
        df["home_team_win"] == 1,
        df["draw"] == 1,
        df["away_team_win"] == 1
    ],
    [2, 1, 0],
    default=-1
)

df["result_name"] = np.select(
    [
        df["home_team_win"] == 1,
        df["draw"] == 1,
        df["away_team_win"] == 1
    ],
    ["home_win", "draw", "away_win"],
    default="unknown"
)

df = df.sort_values(["match_date", "key_id"]).reset_index(drop=True)

columns_to_keep = [
    "key_id", "tournament_id", "tournament_name", "world_cup_year",
    "match_id", "match_name", "stage_name", "group_name",
    "group_stage", "knockout_stage", "replayed", "replay",
    "match_date", "match_time", "stadium_id", "stadium_name",
    "city_name", "country_name",
    "home_team_id", "home_team_name", "home_team_code",
    "away_team_id", "away_team_name", "away_team_code",
    "home_team_score", "away_team_score",
    "goal_difference", "total_goals",
    "extra_time", "penalty_shootout",
    "home_team_score_penalties", "away_team_score_penalties",
    "result", "result_name", "result_label",
    "home_team_win", "away_team_win", "draw"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv("cleanedData.csv", index=False)
print("Dataset limpio:", df_clean.shape)


Dataset limpio: (964, 38)


In [4]:
df_clean = pd.read_csv("cleanedData.csv")
df_clean["match_date"] = pd.to_datetime(df_clean["match_date"], errors="coerce")

def safe_divide(a, b):
    if b == 0:
        return 0
    return a / b

def build_historical_features(matches):
    matches = matches.sort_values(["match_date", "key_id"]).reset_index(drop=True).copy()

    team_stats = {}
    h2h_stats = {}
    featured_rows = []

    def init_team_stats():
        return {
            "matches": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "points": 0,
            "world_cups": set(),
            "recent_goals_for": [],
            "recent_goals_against": [],
            "recent_results": []
        }

    for _, row in matches.iterrows():
        home_id = row["home_team_id"]
        away_id = row["away_team_id"]

        if home_id not in team_stats:
            team_stats[home_id] = init_team_stats()

        if away_id not in team_stats:
            team_stats[away_id] = init_team_stats()

        home_stats = team_stats[home_id]
        away_stats = team_stats[away_id]

        pair_key = tuple(sorted([home_id, away_id]))

        if pair_key not in h2h_stats:
            h2h_stats[pair_key] = {
                "matches": 0,
                home_id: 0,
                away_id: 0,
                "draws": 0
            }

        new_row = row.to_dict()

        # Features históricas antes del partido
        for prefix, stats in [("home", home_stats), ("away", away_stats)]:
            matches_played = stats["matches"]

            new_row[f"{prefix}_historical_matches"] = matches_played
            new_row[f"{prefix}_historical_wins"] = stats["wins"]
            new_row[f"{prefix}_historical_draws"] = stats["draws"]
            new_row[f"{prefix}_historical_losses"] = stats["losses"]
            new_row[f"{prefix}_historical_goals_for"] = stats["goals_for"]
            new_row[f"{prefix}_historical_goals_against"] = stats["goals_against"]
            new_row[f"{prefix}_historical_goal_diff"] = stats["goals_for"] - stats["goals_against"]
            new_row[f"{prefix}_historical_points"] = stats["points"]
            new_row[f"{prefix}_world_cups_played_before"] = len(stats["world_cups"])

            new_row[f"{prefix}_win_rate_before"] = safe_divide(stats["wins"], matches_played)
            new_row[f"{prefix}_draw_rate_before"] = safe_divide(stats["draws"], matches_played)
            new_row[f"{prefix}_loss_rate_before"] = safe_divide(stats["losses"], matches_played)
            new_row[f"{prefix}_avg_goals_for_before"] = safe_divide(stats["goals_for"], matches_played)
            new_row[f"{prefix}_avg_goals_against_before"] = safe_divide(stats["goals_against"], matches_played)
            new_row[f"{prefix}_points_per_match_before"] = safe_divide(stats["points"], matches_played)

            last5_goals_for = stats["recent_goals_for"][-5:]
            last5_goals_against = stats["recent_goals_against"][-5:]
            last5_results = stats["recent_results"][-5:]

            new_row[f"{prefix}_last5_avg_goals_for"] = safe_divide(sum(last5_goals_for), len(last5_goals_for))
            new_row[f"{prefix}_last5_avg_goals_against"] = safe_divide(sum(last5_goals_against), len(last5_goals_against))
            new_row[f"{prefix}_last5_win_rate"] = safe_divide(
                sum(1 for result in last5_results if result == "W"),
                len(last5_results)
            )

        pair_stats = h2h_stats[pair_key]

        new_row["home_vs_away_h2h_matches_before"] = pair_stats["matches"]
        new_row["home_h2h_wins_before"] = pair_stats.get(home_id, 0)
        new_row["away_h2h_wins_before"] = pair_stats.get(away_id, 0)
        new_row["h2h_draws_before"] = pair_stats["draws"]

        featured_rows.append(new_row)

        home_goals = int(row["home_team_score"])
        away_goals = int(row["away_team_score"])
        year = int(row["world_cup_year"])

        if home_goals > away_goals:
            home_result = "W"
            away_result = "L"
            home_points = 3
            away_points = 0
        elif home_goals < away_goals:
            home_result = "L"
            away_result = "W"
            home_points = 0
            away_points = 3
        else:
            home_result = "D"
            away_result = "D"
            home_points = 1
            away_points = 1

        for stats, goals_for, goals_against, result, points in [
            (home_stats, home_goals, away_goals, home_result, home_points),
            (away_stats, away_goals, home_goals, away_result, away_points)
        ]:
            stats["matches"] += 1
            stats["wins"] += int(result == "W")
            stats["draws"] += int(result == "D")
            stats["losses"] += int(result == "L")
            stats["goals_for"] += goals_for
            stats["goals_against"] += goals_against
            stats["points"] += points
            stats["world_cups"].add(year)
            stats["recent_goals_for"].append(goals_for)
            stats["recent_goals_against"].append(goals_against)
            stats["recent_results"].append(result)

        pair_stats["matches"] += 1

        if home_goals > away_goals:
            pair_stats[home_id] += 1
        elif home_goals < away_goals:
            pair_stats[away_id] += 1
        else:
            pair_stats["draws"] += 1

    return pd.DataFrame(featured_rows)

df_featured = build_historical_features(df_clean)

df_featured.to_csv("featuredData.csv", index=False)
print("Dataset con features:", df_featured.shape)

Dataset con features: (964, 78)
